# Phase 3: Multi-Turn GRPO Training

Training the agent **policy** with Reinforcement Learning over full 
multi-turn trajectories — not just single responses.

## What Makes Phase 3 Different From Phase 1

| | Phase 1 | Phase 3 |
|---|---|---|
| What gets trained | Single response | Full trajectory |
| Reward fires on | One output | Entire conversation |
| Model learns | How to reason | When to use tools + when to stop |
| Sampling | 8 responses per prompt | 4 trajectories per problem |

## Key Insight: Credit Assignment
Each trajectory = Think → Write Python → Execute → Read Output → Answer.
The reward must be distributed across all turns that contributed to success.

In [ ]:
!pip install unsloth trl datasets langchain-core

## Reward Function Design

The trajectory reward has 4 components:
```
efficiency_penalty = -0.2 × (num_turns - 1)   # penalize loops
think_reward       = +0.3                       # first turn reasoning
execution_reward   = +1.5                       # correct code execution  
answer_reward      = +0.5 + 2.0                # clean termination + correct answer
```

**Max possible reward:** 4.3 (1-turn, correct, clean)  
**Baseline reward:** ~1.5-1.8 (before training)  
**Target:** reward climbing toward 2.5+

The efficiency penalty is critical — without it, the model learns to 
loop and re-execute correct code to farm execution rewards.

## Training Script

Full multi-turn GRPO training loop written to `phase3_train.py`.

Key design decisions:
- `group_size=4`: Sample 4 full trajectories per problem
- `max_turns=2`: Cap at 2 turns to prevent runaway loops
- `lr=1e-5`: Lower than Phase 1 — policy is already trained, fine-tuning the agent behavior
- Save checkpoint every 10 steps — resume if session crashes
- Distributed setup handles both single GPU and 2-GPU torchrun

In [12]:
%%writefile /kaggle/working/phase3_train.py

import os, re, sys, io, torch, wandb, glob
from unsloth import FastLanguageModel
from datasets import load_dataset
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_

# ── Safe distributed setup ──────────────────────────────────
local_rank = int(os.environ.get("LOCAL_RANK", 0))
world_size = int(os.environ.get("WORLD_SIZE", 1))
is_main = (local_rank == 0)

if world_size > 1:
    import torch.distributed as dist
    dist.init_process_group(backend="nccl")
    torch.cuda.set_device(local_rank)
else:
    torch.cuda.set_device(0)

if is_main:
    print(f"Running with {world_size} GPU(s)")

# ── Resume or start fresh ───────────────────────────────────
checkpoints = sorted(glob.glob("/kaggle/working/phase3_step*"))
if checkpoints:
    MODEL_PATH = checkpoints[-1]
    if is_main: print(f"Resuming from {MODEL_PATH}")
else:
    MODEL_PATH = "/kaggle/input/notebooks/pasha1701/phase1-rl-reasioning-training/grpo_gsm8k/checkpoint-500"
    if is_main: print(f"Starting fresh from {MODEL_PATH}")

# ── Model ───────────────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=2048,
    load_in_4bit=True,
    device_map="auto",
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
if is_main: print("✅ Model loaded!")

# ── System prompt ───────────────────────────────────────────
SYSTEM_PROMPT = """You are an advanced math reasoning assistant.
Think step-by-step inside <think> tags.
For complex calculations, write Python inside ```python``` blocks and use print().
The system will execute your code and return the output.
Once you have the answer, output it inside <answer> tags and stop."""

# ── Tool ────────────────────────────────────────────────────
def safe_execute(code: str) -> str:
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        exec(code, {})
        output = sys.stdout.getvalue().strip()
        sys.stdout = old_stdout
        return output if output else "No output printed."
    except Exception as e:
        sys.stdout = old_stdout
        return f"Error: {e}"

# ── Reward ──────────────────────────────────────────────────
def score_trajectory(model_turns: list, ground_truth: str) -> float:
    reward = 0.0
    reward -= (len(model_turns) - 1) * 0.2
    first_exec_rewarded = False

    for i, turn in enumerate(model_turns):
        is_final = (i == len(model_turns) - 1)

        if i == 0 and "<think>" in turn and "</think>" in turn:
            reward += 0.3

        code_match = re.search(r'```python(.*?)```', turn, re.DOTALL)
        if code_match and not first_exec_rewarded:
            result = safe_execute(code_match.group(1).strip())
            if ground_truth in result:
                reward += 1.5
                first_exec_rewarded = True
            elif "Error" in result:
                reward -= 0.3

        if is_final and "<answer>" in turn:
            reward += 0.5
            ans_match = re.search(r'<answer>(.*?)</answer>', turn, re.DOTALL)
            if ans_match:
                nums = re.findall(r'\d+\.?\d*', ans_match.group(1))
                if nums and nums[-1].strip() == ground_truth:
                    reward += 2.0

    return reward

# ── Trajectory collection ───────────────────────────────────
def generate_turn(messages: list) -> str:
    FastLanguageModel.for_inference(model)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    text += "<think>\n"
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return "<think>\n" + tokenizer.decode(new_tokens, skip_special_tokens=True)


def collect_trajectory(problem: str, ground_truth: str, max_turns=2):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": problem}
    ]
    model_turns = []

    for _ in range(max_turns):
        output = generate_turn(messages)
        model_turns.append(output)
        messages.append({"role": "assistant", "content": output})

        code_match = re.search(r'```python(.*?)```', output, re.DOTALL)
        if code_match:
            result = safe_execute(code_match.group(1).strip())
            messages.append({
                "role": "user",
                "content": f"Terminal Output: {result}\nNow give your final answer in <answer> tags."
            })

        if "<answer>" in output:
            break

    reward = score_trajectory(model_turns, ground_truth)
    return messages, reward


def collect_group(problem: str, ground_truth: str, group_size=4):
    all_messages, rewards = [], []

    for _ in range(group_size):
        msgs, reward = collect_trajectory(problem, ground_truth)
        all_messages.append(msgs)
        rewards.append(reward)

    mean_r = sum(rewards) / len(rewards)
    std_r  = (sum((r - mean_r)**2 for r in rewards) / len(rewards))**0.5 + 1e-8
    advantages = [(r - mean_r) / std_r for r in rewards]

    return all_messages, advantages, rewards

# ── GRPO loss ───────────────────────────────────────────────
def compute_grpo_loss(messages_group: list, advantages: list):
    FastLanguageModel.for_training(model)
    total_loss = torch.tensor(0.0, requires_grad=True).to("cuda")
    valid = 0

    for messages, adv in zip(messages_group, advantages):
        if abs(adv) < 1e-6:
            continue
        full_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        inputs = tokenizer(
            full_text, return_tensors="pt",
            truncation=True, max_length=2048
        ).to("cuda")
        outputs = model(
            input_ids=inputs["input_ids"],
            labels=inputs["input_ids"]
        )
        log_prob = -outputs.loss
        total_loss = total_loss + (-adv * log_prob)
        valid += 1

    return total_loss / valid if valid > 0 else total_loss

# ── Training loop ───────────────────────────────────────────
def train():
    if is_main:
        wandb.login()
        wandb.init(project="my-rlvr-agent", name="phase3-multiturn-grpo")

    dataset = load_dataset("openai/gsm8k", "main", split="train")
    dataset = dataset.shuffle(seed=42)

    optimizer = AdamW(model.parameters(), lr=1e-5)

    NUM_STEPS  = 80
    GROUP_SIZE = 4

    if is_main:
        print(f"\n🚀 Phase 3 | Steps: {NUM_STEPS} | Group: {GROUP_SIZE}\n")

    running_rewards = []

    for step, example in enumerate(dataset):
        if step >= NUM_STEPS:
            break

        problem = example["question"]
        gt      = example["answer"].split("####")[-1].strip().replace(",", "")

        try:
            messages_group, advantages, rewards = collect_group(
                problem, gt, GROUP_SIZE
            )
        except Exception as e:
            if is_main: print(f"⚠️  Step {step} failed: {e}")
            continue

        mean_reward = sum(rewards) / len(rewards)
        running_rewards.append(mean_reward)

        optimizer.zero_grad()
        loss = compute_grpo_loss(messages_group, advantages)
        if loss.requires_grad:
            loss.backward()
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # Save every 10 steps
        if is_main and (step + 1) % 10 == 0:
            avg = sum(running_rewards[-10:]) / min(len(running_rewards), 10)
            print(f"Step {step+1:3d}/{NUM_STEPS} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Avg Reward: {avg:.3f}")
            wandb.log({
                "step": step + 1,
                "loss": loss.item(),
                "mean_reward": mean_reward,
                "avg_reward": avg,
            })
            # Save checkpoint
            path = f"/kaggle/working/phase3_step{step+1}"
            model.save_pretrained(path)
            tokenizer.save_pretrained(path)
            print(f"💾 Saved: {path}")

    if is_main:
        model.save_pretrained("/kaggle/working/phase3_final")
        tokenizer.save_pretrained("/kaggle/working/phase3_final")
        print("\n✅ Done!")
        wandb.finish()

train()

Overwriting /kaggle/working/phase3_train.py


## Launch Training

Using torchrun with 2x T4 GPUs.  
Training runs for 80 steps (~6-8 hours).  
Checkpoints saved every 10 steps to `/kaggle/working/phase3_stepN`.

In [14]:
import os
os.environ["WANDB_API_KEY"] = "wandb_v1_"

!torchrun \
    --nproc_per_node=2 \
    --master_port=29500 \
    /kaggle/working/phase3_train.py

W0312 05:39:30.870000 368 torch/distributed/run.py:852] 
W0312 05:39:30.870000 368 torch/distributed/run.py:852] *****************************************
W0312 05:39:30.870000 368 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0312 05:39:30.870000 368 torch/distributed/run.py:852] *****************************************
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-12 05:39:36.829185: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773293976.856360     374 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for 

## Training Results

Reward curve over 80 steps:

| Step | Avg Reward |
|------|-----------|
| 10   | 1.855     |
| 20   | 1.512     |
| 30   | 1.485     |
| 40   | 2.197     |
| 50   | 2.382     |
| 60   | 2.353     |
| 70   | **2.540** ← peak |
| 80   | 1.310     |

Best checkpoint: **Step 70** (reward 2.540)  
Reward improved from 1.855 → 2.540 — agent learned more efficient trajectories.

Now testing the trained agent on a complex problem.

## Phase 3 Agent — Live Inference Test

Testing the trained agent on problems **outside the training distribution**.
The model was trained on GSM8K math — these are novel problems it has never seen.

This tests whether Phase 3 RL training generalized to:
- Complex multi-step calculations
- Non-standard formula applications  
- Dynamic cost/pricing problems

The agent should: reason in `<think>` → write Python → execute → give `<answer>`

In [15]:
import sys, io, re, torch
from unsloth import FastLanguageModel

# 1. LOAD YOUR TRAINED AGENT (Taking only 5-10 seconds)
print("Loading Phase 3 Agent (Step 50)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/phase3_step50", # Using your best checkpoint!
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("✅ Agent Ready!\n")

# 2. THE PYTHON TOOL
def safe_execute(code: str) -> str:
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        exec(code, {})
        output = sys.stdout.getvalue().strip()
        sys.stdout = old_stdout
        return output if output else "No output printed."
    except Exception as e:
        sys.stdout = old_stdout
        return f"Error: {e}"

# 3. THE AGENTIC LOOP
def ask_agent(problem: str, max_turns=3):
    SYSTEM_PROMPT = """You are an advanced math reasoning assistant.
Think step-by-step inside <think> tags.
For complex calculations, write Python inside ```python``` blocks and use print().
The system will execute your code and return the output.
Once you have the answer, output it inside <answer> tags and stop."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem}
    ]
    
    print(f"📝 PROBLEM: {problem}\n" + "="*50)
    
    for turn in range(max_turns):
        print(f"\n🔄 Turn {turn + 1}...")
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        text += "<think>\n"
        inputs = tokenizer([text], return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=500, temperature=0.2,
                pad_token_id=tokenizer.eos_token_id, use_cache=True
            )
        
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response = "<think>\n" + tokenizer.decode(new_tokens, skip_special_tokens=True)
        print(response)
        
        messages.append({"role": "assistant", "content": response})
        
        if "<answer>" in response:
            print("\n✅ Agent has provided the final answer. Stopping.")
            break
            
        code_match = re.search(r'```python(.*?)```', response, re.DOTALL)
        if code_match:
            code = code_match.group(1).strip()
            print("\n⚙️  EXECUTING AGENT'S CODE:\n" + "-"*30 + f"\n{code}\n" + "-"*30)
            
            result = safe_execute(code)
            print(f"🖥️  TERMINAL OUTPUT: {result}")
            
            messages.append({
                "role": "user", 
                "content": f"Terminal Output: {result}\nNow provide your final answer in <answer> tags."
            })
            
    return messages

# ==========================================
# 🧪 THE "SMALL" 15-SECOND TEST
# ==========================================
test_problem = "A factory produces items where the production cost of the N-th item is given by the formula: Cost = 5 + (N * 0.15) ** 1.2. Write a Python script to calculate the exact total cost to produce the first 5,000 items. Print the total sum rounded to 2 decimal places, and output that final number in your answer tags."

# This will only take a few seconds!
conversation = ask_agent(test_problem)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-12 07:31:01.510176: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773300661.535858     237 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773300661.546589     237 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773300661.571714     237 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773300661.571740     237 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773300661.571743     237 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Phase 3 Agent (Step 50)...
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.17.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Agent Ready!

📝 PROBLEM: A factory produces items where the production cost of the N-th item is given by the formula: Cost = 5 + (N * 0.15) ** 1.2. Write a Python script to calculate the exact total cost to produce the first 5,000 items. Print the total sum rounded to 2 decimal places, and output that final number in your answer tags.

🔄 Turn 1...
<think>
To solve this problem, we need to calculate the cost for each item from 1 to 5000 using the provided formula and then sum these costs.

Here's how we can do it:

1. Initialize a variable `total_cost` to store the cumulative cost.
2. Loop through numbers from 1 to 5000, calculating the cost for each item using the formula.
3. Add the calculated cost to `total_cost`.
4. After the loop, round the total cost to two decimal places.
</think>

```python
# Initialize total cost
total_cost = 0

# Calculate the total cost for producing the first 5000 items
for i in range(1, 5001):
    # Calculate the cost for the current item
    cost = 5 + (

## Inference Results

The Phase 3 agent correctly solved complex out-of-distribution problems:

- ✅ Used `<think>` tags for structured reasoning
- ✅ Wrote correct Python code for the iterative calculation  
- ✅ Executed code and read terminal output correctly
- ✅ Terminated cleanly with `<answer>` tags in 2 turns

**Key finding:** The efficiency penalty in the reward function worked — 
the agent solved in minimal turns rather than looping unnecessarily.

This demonstrates the agent learned genuine tool-use behavior, 
not just pattern matching on GSM8K problems.

## Deploy to HuggingFace Hub

Merging LoRA adapters into the base model and pushing as a 
standalone model for public use.

Merging converts: `base_model + LoRA weights → single deployable model`

Published at: [mpasha1701/RLVR-Qwen2.5-1.5B-Agent](https://huggingface.co/mpasha1701/RLVR-Qwen2.5-1.5B-Agent)

In [16]:
from unsloth import FastLanguageModel

# 1. Load your best checkpoint (Ensure load_in_4bit is False for merging)
print("Loading Step 50 adapters for merging...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/phase3_step50", 
    max_seq_length=2048,
    load_in_4bit=False, 
)

# 2. Your HF Details
HF_TOKEN = "hf_"
REPO_NAME = "mpasha1701/RLVR-Qwen2.5-1.5B-Agent" 

# 3. Merge the Base Model + LoRA and Push to Hugging Face
print(f"Merging weights and pushing to {REPO_NAME}...")
model.push_to_hub_merged(
    REPO_NAME, 
    tokenizer, 
    save_method="merged_16bit", 
    token=HF_TOKEN
)

print("🚀 Upload complete! Your standalone model is live.")

Loading Step 50 adapters for merging...
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.17.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Merging weights and pushing to mpasha1701/RLVR-Qwen2.5-1.5B-Agent...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:56<00:00, 56.16s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/mpasha1701/RLVR-Qwen2.5-1.5B-Agent`
🚀 Upload complete! Your standalone model is live.
